In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
df= pd.read_csv("creditcard.csv")

In [3]:
X= df.drop(columns=["Time","Class"])
y=df["Class"]

In [4]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64

In [5]:
X.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99


In [6]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
pipeline= Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced' , max_iter=10000, random_state=42)
    ),
])

pipeline.fit(X_train, y_train)
probas = pipeline.predict_proba(X_test)[:,1]
test_amount = X_test['Amount'].values

In [36]:
def caculate_loss(y_true, probas, amounts, threshold ):
    preds = (probas>=threshold).astype(int)
    fn_mask= (preds==0) & (y_true==1)
    fn_loss= np.sum(amounts[fn_mask])

    fp_mask=(preds==1) & (y_true == 0)
    fp_loss=np.sum(fp_mask)*10
    total_loss = fn_loss + fp_loss
    return total_loss, fn_loss, fp_loss
    

In [40]:

thresholds = np.linspace(0.01, 0.99, 99)
results = [
    caculate_loss(
        y_test.values, probas, test_amount, t
    )
    for t in thresholds
]
losses = [r[0] for r in results]

best_idx = np.argmin(losses)
best_threshold = thresholds[best_idx]
optimal_loss = losses[best_idx]
default_loss = caculate_loss(
    y_test.values, probas, test_amount, 0.5
)[0]

print('\n--- Financial Impact Analysis ---')
print(f'Total Test Fraud Amount: ${np.sum(test_amount[y_test == 1]):,.2f}')
print(f'Loss at Default Threshold (0.50): ${default_loss:,.2f}')
print(
    f'Loss at Optimal Threshold ({best_threshold:.2f}): ${optimal_loss:,.2f}'
)
print(f'💵 Net Savings: ${default_loss - optimal_loss:,.2f}')

# 6. Save Model Artifacts
joblib.dump(
    {
        'pipeline': pipeline,
        'best_threshold': best_threshold,
        'feature_names': list(X.columns),
    },
    'fraud_detection_model.joblib',
)
print('\nModel artifact saved as "fraud_detection_model.joblib"!')


--- Financial Impact Analysis ---
Total Test Fraud Amount: $16,078.40
Loss at Default Threshold (0.50): $15,459.57
Loss at Optimal Threshold (0.99): $4,152.58
💵 Net Savings: $11,306.99

Model artifact saved as "fraud_detection_model.joblib"!
